In [ ]:
# parameters
# BINDER_FAST: set phi_search_pts=50 for fast cloud execution
beta = 0.1              # junction asymmetry ratio (small/large E_J)
M = 3                   # number of large junctions in shunting array
phi_e = 0.4 * 2 * 3.141592653589793   # external flux bias (radians)
E_J = 10.0              # large-junction Josephson energy (GHz)
phi_search_pts = 200    # grid points for coarse minimum search

In [ ]:
# hide
import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget

from bosonic_gates.snail import (
    snail_potential,
    find_snail_minimum,
    snail_taylor_coefficients,
    plot_snail_potential,
)

## Module 2a: The SNAIL Potential and Taylor Coefficients

**Learning objectives:**
- Write down and evaluate the SNAIL potential $U_s(\phi)$
- Find the potential minimum $\phi_m$ numerically
- Extract the Taylor coefficients $c_2$–$c_5$ and understand their physical meaning
- Observe how the cubic $c_3$ and quartic $c_4$ respond to flux tuning

---

**Sections:** [1 The SNAIL Circuit](#sec1) · [2 Potential Landscape](#sec2) · [3 Taylor Coefficients](#sec3) · [4 Sweeping Flux](#sec4)

<a id="sec1"></a>
## 1  The SNAIL Circuit

The **SNAIL** (Superconducting Nonlinear Asymmetric Inductive eLement) was introduced by Frattini *et al.* (2017) and systematically characterised by Chapman *et al.*, *PRX Quantum* **4**, 020355 (2023). It consists of a single small Josephson junction with energy $\beta E_J$ connected in a superconducting loop with an array of $M$ larger junctions, each with energy $E_J$. An external magnetic flux $\Phi_e$ threads the loop.

The key design lever is the **asymmetry ratio** $\beta < 1/M$: making the small junction weaker than the array creates a *single* potential minimum in the loop (unlike a standard rf-SQUID which can have multiple). This ensures the circuit is a true single-mode oscillator.

The total potential energy (summing Josephson energies around the loop, enforcing flux quantisation) is (Chapman *et al.* Eq. B4):

$$\boxed{U_s(\phi) = E_J\!\left[-\beta\cos(\phi - \phi_e) - M\cos\!\left(\frac{\phi}{M}\right)\right]}$$

where $\phi$ is the phase across the small junction and $\phi_e = 2\pi\,\Phi_e/\Phi_0$ is the dimensionless external flux bias.

**Why $\beta < 1/M$?** Differentiating once: $dU_s/d\phi = E_J[\beta\sin(\phi-\phi_e) + \sin(\phi/M)]$. For $\beta < 1/M$ this equation has a unique zero per $2\pi M$ interval — a single minimum. If $\beta \geq 1/M$ the small-junction cosine can dominate and produce multiple competing minima, making the circuit multistable.

*Lab note: Typical SNAIL parameters are $M = 3$, $\beta \approx 0.1$, and $E_J \approx 10$–$20\,\text{GHz}$. The flux bias $\phi_e \approx 0.4\times 2\pi$ is chosen to maximise the cubic nonlinearity (see Section 4).*

<a id="sec2"></a>
## 2  Potential Landscape

We evaluate $U_s(\phi)$ over a wide phase range to see the periodic structure and locate the primary minimum.

In [ ]:
# Evaluate the SNAIL potential over [-4pi, 4pi]
phi = np.linspace(-4 * np.pi, 4 * np.pi, 2000)
U = snail_potential(phi, beta=beta, M=M, phi_e=phi_e, E_J=E_J)

# Find the potential minimum
phi_m = find_snail_minimum(beta, M, phi_e)
print(f"Potential minimum: phi_m = {phi_m:.4f} rad  ({phi_m/np.pi:.4f} pi)")
print(f"U(phi_m) = {snail_potential(phi_m, beta=beta, M=M, phi_e=phi_e, E_J=E_J):.4f} GHz")

# Plot
fig, ax = plot_snail_potential(phi, U / E_J, phi_m=phi_m)
ax.set_title(rf"SNAIL potential: $\beta={beta}$, $M={M}$, $\phi_e={phi_e/(2*np.pi):.2f}\times 2\pi$")
plt.show()

In [ ]:
# Overlay three flux values to show flux tuning
phi_e_vals = [0.3 * 2 * np.pi, 0.4 * 2 * np.pi, 0.5 * 2 * np.pi]
labels_flux = [rf"$\phi_e = {v/(2*np.pi):.1f}\times 2\pi$" for v in phi_e_vals]
colors_flux = ["C0", "C1", "C2"]

U_list = [
    snail_potential(phi, beta=beta, M=M, phi_e=pe, E_J=E_J) / E_J
    for pe in phi_e_vals
]
phi_m_list = [find_snail_minimum(beta, M, pe) for pe in phi_e_vals]

fig2, ax2 = plot_snail_potential(
    phi, U_list,
    phi_m=phi_m_list,
    labels=labels_flux,
    colors=colors_flux,
    xlim=(-2 * np.pi, 2 * np.pi),
)
ax2.set_title(rf"Flux tuning: $\beta={beta}$, $M={M}$")
plt.show()

print("Minimum positions (units of pi):")
for pe, pm in zip(phi_e_vals, phi_m_list):
    print(f"  phi_e = {pe/(2*np.pi):.1f}*2pi  =>  phi_m = {pm/np.pi:.4f} pi")

*Lab note: As $\phi_e$ is tuned from $0$ to $\pi$, the minimum $\phi_m$ shifts and the asymmetry of the well changes. The asymmetry (odd Taylor coefficients, $c_3$, $c_5$) is maximised near half-integer flux $\phi_e \approx \pi$. At $\phi_e = 0$ or $\phi_e = 2\pi$ the SNAIL is symmetric and all odd coefficients vanish.*

<a id="sec3"></a>
## 3  Taylor Coefficients

Near the minimum $\phi_m$, we expand in the displaced phase $\varphi = \phi - \phi_m$:

$$\frac{U_s(\phi_m + \varphi)}{E_J} \approx \frac{c_2}{2!}\varphi^2 + \frac{c_3}{3!}\varphi^3 + \frac{c_4}{4!}\varphi^4 + \frac{c_5}{5!}\varphi^5 + \cdots$$

The coefficients are the successive derivatives of $U_s/E_J$ evaluated at the minimum (Chapman *et al.* Eq. B6–B7):

$$c_2 = \beta\cos(\phi_m - \phi_e) + \frac{1}{M}\cos\!\left(\frac{\phi_m}{M}\right)$$

$$c_3 = \frac{M^2-1}{M^2}\sin\!\left(\frac{\phi_m}{M}\right)$$

$$c_4 = -\beta\cos(\phi_m-\phi_e) - \frac{1}{M^3}\cos\!\left(\frac{\phi_m}{M}\right)$$

$$c_5 = \frac{1-M^4}{M^4}\sin\!\left(\frac{\phi_m}{M}\right)$$

**Sign convention.** All $c_n$ here are coefficients in the Taylor series of $U_s/E_J$, which is dimensionless. The physical curvature of the well is $E_J c_2 > 0$ (necessary for stability). The cubic $c_3$ is odd in $\phi_e - \pi$ and is maximised near half-flux: this is the key parameter controlling three-wave mixing. Note also that $c_3$ depends only on $\phi_m$ (not $\beta$), while $c_2$ and $c_4$ contain both the small-junction ($\beta$) and array contributions.

*Lab note: The ratio $c_3/c_2$ sets the dimensionless cubic strength per unit well curvature. A large $c_3/c_2$ means stronger anharmonicity for the same oscillator frequency — the hallmark of a good SNAIL operating point.*

In [ ]:
# Compute Taylor coefficients at the chosen operating point
coeffs = snail_taylor_coefficients(beta, M, phi_e)

print("Taylor coefficients (dimensionless, of U_s/E_J):")
for key in ['c2', 'c3', 'c4', 'c5']:
    print(f"  {key} = {coeffs[key]:.6f}")
print(f"  phi_m = {coeffs['phi_m']:.6f} rad  ({coeffs['phi_m']/np.pi:.4f} pi)")
print()
print(f"Cubic-to-quadratic ratio  |c3/c2| = {abs(coeffs['c3']/coeffs['c2']):.4f}")
print(f"Quartic sign: c4 = {coeffs['c4']:.6f}  ({'negative (Kerr-like)' if coeffs['c4'] < 0 else 'positive'})")

In [ ]:
# Visualise the Taylor approximation vs the exact potential
dphi = np.linspace(-1.5, 1.5, 500)   # displacement from minimum
c2, c3, c4, c5 = coeffs['c2'], coeffs['c3'], coeffs['c4'], coeffs['c5']

U_exact   = snail_potential(phi_m + dphi, beta=beta, M=M, phi_e=phi_e, E_J=E_J) / E_J
U_exact  -= U_exact[len(dphi)//2]      # shift minimum to zero

U_quad  = c2/2  * dphi**2
U_cubic = U_quad  + c3/6  * dphi**3
U_quart = U_cubic + c4/24 * dphi**4
U_quint = U_quart + c5/120* dphi**5

fig3, ax3 = plt.subplots(figsize=(6, 4))
ax3.plot(dphi, U_exact,  lw=2,   label="exact",          color="k")
ax3.plot(dphi, U_quad,   lw=1.5, label=r"$c_2$ only",   ls="--", color="C0")
ax3.plot(dphi, U_cubic,  lw=1.5, label=r"$+c_3$",       ls="-.", color="C1")
ax3.plot(dphi, U_quart,  lw=1.5, label=r"$+c_4$",       ls=":",  color="C2")
ax3.plot(dphi, U_quint,  lw=1.5, label=r"$+c_5$",       ls="-",  color="C3", alpha=0.7)
ax3.set_xlabel(r"Displacement $\delta\phi = \phi - \phi_m$")
ax3.set_ylabel(r"$U_s / E_J$ (shifted)")
ax3.set_title("Taylor approximation vs exact potential")
ax3.set_xlim(-1.5, 1.5)
ax3.set_ylim(-0.1, 0.8)
ax3.legend(fontsize=8, frameon=False)
ax3.tick_params(direction="in", top=True, right=True)
plt.tight_layout()
plt.show()

<a id="sec4"></a>
## 4  Sweeping Flux

All four Taylor coefficients depend on $\phi_e$ through the minimum position $\phi_m(\phi_e)$. Sweeping the external flux gives full *in-situ* control of the potential shape.

In [ ]:
# Sweep external flux from 0.1 to 0.9 * 2pi (avoid endpoints where min finder is less stable)
phi_e_arr = np.linspace(0.1 * 2 * np.pi, 0.9 * 2 * np.pi, 80)

c2_arr = np.zeros(len(phi_e_arr))
c3_arr = np.zeros(len(phi_e_arr))
c4_arr = np.zeros(len(phi_e_arr))
c5_arr = np.zeros(len(phi_e_arr))

for i, pe in enumerate(phi_e_arr):
    co = snail_taylor_coefficients(beta, M, pe)
    c2_arr[i] = co['c2']
    c3_arr[i] = co['c3']
    c4_arr[i] = co['c4']
    c5_arr[i] = co['c5']

# Plot
x_flux = phi_e_arr / (2 * np.pi)

fig4, axes4 = plt.subplots(2, 2, figsize=(8, 5))
specs = [
    (axes4[0, 0], c2_arr, r"$c_2$",  "C0"),
    (axes4[0, 1], c3_arr, r"$c_3$",  "C1"),
    (axes4[1, 0], c4_arr, r"$c_4$",  "C2"),
    (axes4[1, 1], c5_arr, r"$c_5$",  "C3"),
]
for ax, arr, lbl, col in specs:
    ax.plot(x_flux, arr, lw=1.5, color=col)
    ax.axhline(0, color='k', lw=0.5, ls='--')
    ax.set_xlabel(r"$\Phi_e/\Phi_0$")
    ax.set_ylabel(lbl)
    ax.tick_params(direction='in', top=True, right=True)

# Mark operating point phi_e = 0.4 * 2pi
for ax, _, _, _ in specs:
    ax.axvline(phi_e / (2 * np.pi), color='crimson', ls=':', lw=1.0, alpha=0.7,
               label=rf"$\phi_e={phi_e/(2*np.pi):.2f}$")

axes4[0, 1].legend(fontsize=7, frameon=False)
fig4.suptitle(rf"SNAIL Taylor coefficients vs flux, $\beta={beta}$, $M={M}$", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Locate the maximum of |c3| and where c4 changes sign (Kerr-free point preview)
idx_c3_max = int(np.argmax(np.abs(c3_arr)))
print(f"Maximum |c3|: phi_e = {x_flux[idx_c3_max]:.3f} * 2pi  =>  c3 = {c3_arr[idx_c3_max]:.5f}")

# Find zero crossings of c4
sign_changes = np.where(np.diff(np.sign(c4_arr)))[0]
print("c4 sign changes (Kerr-free flux candidates):")
for idx in sign_changes:
    pe_kf = np.interp(0, [c4_arr[idx], c4_arr[idx+1]], [x_flux[idx], x_flux[idx+1]])
    print(f"  phi_e ~ {pe_kf:.3f} * 2pi")

*Lab note: The cubic coefficient $c_3$ peaks near $\Phi_e/\Phi_0 \approx 0.4$–$0.45$ for $M=3$, $\beta=0.1$. At this flux point the quartic $c_4$ is still nonzero (negative, Kerr-like). The flux where $c_4 = 0$ is the **Kerr-free point** — a critical operating condition explored in Notebook 2b. Both $c_3$ and $c_5$ vanish at integer and half-integer flux (where the well is symmetric), and $c_2$ varies smoothly, setting the oscillator frequency.*